# GeoDeep - Object Detection & Segmentation

[![image](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/opengeos/geoai/blob/main/docs/examples/geodeep_detection.ipynb)

This notebook demonstrates how to use [GeoDeep](https://github.com/uav4geo/GeoDeep) integration in GeoAI for object detection and semantic segmentation on geospatial imagery.

GeoDeep is a lightweight library that applies ONNX AI models to GeoTIFF rasters. Key features:

- **Object Detection**: Cars, trees, birds, planes, utilities, and multi-class aerial detection
- **Semantic Segmentation**: Buildings and roads
- **GPU accelerated**: Automatically uses CUDA GPU if available via ONNX Runtime, falls back to CPU
- **Georeferenced output**: Results in EPSG:4326 as GeoDataFrame, GeoJSON, or GeoTIFF masks

## Installation

Uncomment the following line to install the required packages.

In [ ]:
# %pip install -U "geoai-py[geodeep]"

In [ ]:
import geoai

device = geoai.get_device()
print(f"Using device: {device}")

## List Available Models

GeoDeep includes 9 built-in models for detection and segmentation tasks. Models are automatically downloaded and cached on first use.

In [ ]:
from geoai import list_geodeep_models, GEODEEP_MODELS

for name, desc in list_geodeep_models().items():
    model_type = GEODEEP_MODELS[name]["type"]
    print(f"  {name:15s} [{model_type:13s}] {desc}")

## Download Sample Data

Download a high-resolution aerial image for testing. Replace the URL below with your own GeoTIFF if desired.

In [ ]:
url = "https://data.source.coop/opengeos/geoai/cars-7cm.tif"
image_path = geoai.download_file(url)

## Visualize the Input Image

In [ ]:
geoai.view_raster(url)

## Object Detection

### Car Detection

Use the `GeoDeep` class to detect cars in the aerial image. The result is a GeoDataFrame with bounding box polygons, confidence scores, and class labels.

In [ ]:
from geoai import GeoDeep

gd = GeoDeep("cars")
print(gd)

In [ ]:
detections = gd.detect(image_path)
print(f"Found {len(detections)} cars")
detections.head()

### Visualize Detections

Overlay detection results on the original image.

In [ ]:
geoai.view_vector_interactive(detections, tiles=image_path)

### Save Detection Results

Export detections to GeoJSON, GeoPackage, or Shapefile.

In [ ]:
gd.detect(image_path, output_path="car_detections.geojson")

### Detection with Confidence Filtering

Use a higher confidence threshold to reduce false positives.

In [ ]:
gd_strict = GeoDeep("cars", conf_threshold=0.7)
high_conf = gd_strict.detect(image_path)
print(f"High-confidence detections: {len(high_conf)}")
high_conf.head()

### Tree Detection

In [ ]:
url = "https://data.source.coop/opengeos/geoai/uc-berkeley.tif"
image_path = geoai.download_file(url)

In [ ]:
gd_trees = GeoDeep("trees")
trees = gd_trees.detect(image_path)
print(f"Found {len(trees)} trees")
trees.head()

In [ ]:
geoai.view_vector_interactive(trees, tiles=image_path)

### Model Properties

Access model metadata via OOP properties.

In [ ]:
gd_aero = GeoDeep("aerovision")
print(f"Model type:  {gd_aero.model_type}")
print(f"Classes:     {gd_aero.available_classes}")
print(f"Description: {gd_aero.model_info['description']}")
print(f"Resolution:  {gd_aero.model_info['resolution']}")

## Semantic Segmentation

### Building Segmentation

Segment buildings and save the result as a GeoTIFF mask.

In [ ]:
gd_buildings = GeoDeep("buildings")
print(gd_buildings)
print(f"Classes: {gd_buildings.available_classes}")

In [ ]:
result = gd_buildings.segment(
    image_path,
    output_raster_path="buildings_mask.tif",
)
print(f"Mask shape: {result['mask'].shape}")
print(f"Unique values: {set(result['mask'].flatten())}")

### Segmentation to Vector

Export segmentation as vector polygons (GeoPackage).

In [ ]:
result = gd_buildings.segment(
    image_path,
    output_vector_path="buildings.gpkg",
)
buildings_gdf = result["gdf"]
print(f"Found {len(buildings_gdf)} building polygons")
buildings_gdf.head()

In [ ]:
geoai.view_vector_interactive(buildings_gdf, tiles=image_path)

### Road Segmentation

In [ ]:
gd_roads = GeoDeep("roads")
road_result = gd_roads.segment(
    image_path,
    output_raster_path="roads_mask.tif",
    output_vector_path="roads.gpkg",
)
roads_gdf = road_result["gdf"]
print(f"Road mask shape: {road_result['mask'].shape}")
print(f"Found {len(roads_gdf)} road polygons")

In [ ]:
geoai.view_vector_interactive(roads_gdf, tiles=url)

## Convenience Functions

For one-off calls, use module-level convenience functions without creating a class instance.

In [ ]:
from geoai import geodeep_detect, geodeep_segment

# One-liner detection
plane_detections = geodeep_detect(image_path, model_id="planes")
print(f"Planes found: {len(plane_detections)}")

# One-liner segmentation
seg_result = geodeep_segment(
    image_path,
    model_id="buildings",
    output_raster_path="quick_buildings.tif",
)
print(f"Segmentation mask shape: {seg_result['mask'].shape}")

## Batch Processing

Process multiple images at once. Results are combined into a single GeoDataFrame with a `source_file` column.

In [ ]:
# Example with a list of images (using the same image twice for demo)
image_list = [image_path, image_path]

gd_batch = GeoDeep("cars")
batch_results = gd_batch.detect_batch(image_list, output_dir="batch_results/")
print(f"Total detections across all images: {len(batch_results)}")
batch_results[["class", "score", "source_file"]].head()